# Memory Projects

**Module:** 13 — AI Memory

Hands-on builds: preference agent, hybrid journal, multi-tenant memory API.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Build end-to-end mini projects that exercise capture, storage, and retrieval
- Apply tenant isolation and budgeting in a small API
- Document acceptance tests for memory behavior


## Project 1 — Personal Preference Memory Agent

**Goal:** An agent that stores preferences from chat and uses them on later turns.

### Requirements
| ID | Requirement |
|----|-------------|
| P1-1 | Detect preference utterances (`prefer`, `always`, `never`) |
| P1-2 | Upsert into a per-user store |
| P1-3 | Inject top preferences into the system prompt (budget 300 chars) |
| P1-4 | `/memories` lists what is stored; `/forget key` deletes |

```mermaid
flowchart LR
  U[User] --> A[Agent]
  A --> D{Preference?}
  D -->|yes| W[Upsert]
  D -->|no| R[Retrieve+Answer]
  W --> R
```


In [ ]:
# Project 1 starter
import re
from dataclasses import dataclass, field

@dataclass
class PrefAgent:
    prefs: dict[str, str] = field(default_factory=dict)

    def maybe_capture(self, text: str) -> str | None:
        m = re.search(r"i (?:prefer|always|never) (.+)", text.lower())
        if not m:
            return None
        val = m.group(0)
        key = "style" if "prefer" in val or "always" in val else "constraint"
        self.prefs[key] = text.strip()
        return key

    def prompt(self, user: str, budget=300) -> str:
        block = []
        used = 0
        for k, v in self.prefs.items():
            line = f"- ({k}) {v}"
            if used + len(line) > budget:
                break
            block.append(line)
            used += len(line)
        mem = "\n".join(block) or "(none)"
        return f"You are helpful.\n## Memory\n{mem}\n## User\n{user}"

agent = PrefAgent()
print("captured", agent.maybe_capture("I prefer concise bullets"))
print("captured", agent.maybe_capture("What is 2+2?"))
print(agent.prompt("How should you answer?"))


### Try it yourself — Project 1

1. Add entity-keyed prefs (`timezone`, `name`, `tone`) with simple parsers.
2. Write 5 assertion checks for capture/inject/forget.

**Stretch:** Use an LLM JSON extractor mock that returns `{"key","value","importance"}`.


## Project 2 — Hybrid Semantic + Episodic Journal

**Goal:** Daily journal agent that stores episodes and extracts semantic facts nightly.

### Pipeline walkthrough
1. User writes journal entry (episode)  
2. Store raw + embedding  
3. Nightly job: extract facts (`mood`, `people`, `commitments`)  
4. Morning brief: retrieve last 3 episodes + open commitments  

| Acceptance | Test |
|------------|------|
| Episode retained | After write, search finds it |
| Fact extracted | Commitment appears in semantic store |
| Temporal filter | "yesterday" excludes older |


In [ ]:
# Project 2 starter — episode + semantic extraction
from datetime import date

episodes = []
semantics = {}

def add_episode(text: str, day: date):
    episodes.append({"day": day, "text": text})

def extract_commitments(text: str):
    if "i will" in text.lower():
        semantics.setdefault("commitments", []).append(text)

add_episode("Meeting with Ava. I will send the deck.", date(2026, 8, 1))
add_episode("Gym day", date(2026, 8, 1))
for e in episodes:
    extract_commitments(e["text"])
print("episodes", len(episodes), "commitments", semantics)


## Project 3 — Multi-Tenant Memory Service API

**Goal:** Tiny HTTP-shaped service (functions are fine) with tenant isolation.

### Endpoints (shapes)
```json
POST /v1/{tenant_id}/memories {"user_id":"u","type":"fact","text":"..."}
GET  /v1/{tenant_id}/memories/search?user_id=u&q=...&k=5
DELETE /v1/{tenant_id}/memories/{id}
```

### Non-negotiables
- Cross-tenant search returns empty / 404 never leaks  
- Auth stub checks `Authorization: Bearer YOUR_API_KEY`  
- Every record includes `tenant_id`


In [ ]:
# Project 3 starter
import hashlib
from dataclasses import dataclass

API_KEY = "YOUR_API_KEY"

@dataclass
class Mem:
    id: str
    tenant_id: str
    user_id: str
    text: str

class MemoryService:
    def __init__(self):
        self.rows: list[Mem] = []

    def _auth(self, bearer: str):
        if bearer != f"Bearer {API_KEY}" and API_KEY != "YOUR_API_KEY":
            # In classroom mode YOUR_API_KEY accepts the placeholder bearer
            pass
        if not bearer.startswith("Bearer "):
            raise PermissionError("unauthorized")

    def create(self, bearer, tenant_id, user_id, text):
        self._auth(bearer)
        mid = hashlib.sha256(f"{tenant_id}:{user_id}:{text}".encode()).hexdigest()[:10]
        self.rows.append(Mem(mid, tenant_id, user_id, text))
        return mid

    def search(self, bearer, tenant_id, user_id, q, k=5):
        self._auth(bearer)
        qset = set(q.lower().split())
        hits = []
        for r in self.rows:
            if r.tenant_id != tenant_id or r.user_id != user_id:
                continue
            score = len(qset & set(r.text.lower().split()))
            if score:
                hits.append((score, r))
        hits.sort(reverse=True)
        return [r for _, r in hits[:k]]

svc = MemoryService()
auth = "Bearer YOUR_API_KEY"
svc.create(auth, "t1", "u1", "likes tea")
svc.create(auth, "t2", "u1", "SECRET")
print([m.text for m in svc.search(auth, "t1", "u1", "tea")])
assert svc.search(auth, "t1", "u1", "SECRET") == []
print("isolation ok")


## Deliverables Checklist (all projects)

- [ ] README section in notebook cells describing architecture  
- [ ] At least 3 automated asserts / tests  
- [ ] Diagram (mermaid or ASCII)  
- [ ] List of failure modes you handled  
- [ ] Placeholder env vars documented  

### Portfolio tip
Record a short screencast: preference saved in session A, recalled in session B, forgotten on request.


### Try it yourself — Ship one

1. Finish Project 1 with forget + list commands.
2. Add a cross-tenant adversarial test to Project 3.

**Stretch:** Containerize Project 3 mentally: what is stateful? how do you backup memories?


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `acceptance test` | Executable check of a requirement |
| `multi-tenant` | One system serving isolated customers |
| `write path` | Code that persists new memories |


## Project Rubric (apply to all three)

| Criterion | Weight | Excellent |
|-----------|--------|-----------|
| Correctness / tests | 30% | Isolation + budget asserts |
| Design clarity | 20% | Diagram + schema |
| UX commands | 15% | list/forget/remember |
| Observability | 15% | logs with ids |
| Write-up | 20% | failure modes + next steps |

### Demo script (Project 1)
1. Set preference in turn 1  
2. New `ShortTermMemory` session object (simulate new session) with same LTM  
3. Ask "how should you answer?" → uses preference  
4. `/forget style` → preference gone  


In [ ]:
# Rubric auto-smoke for project 3 isolation
from hashlib import sha256

class TinyAPI:
    def __init__(self):
        self.rows = []
    def add(self, tenant, user, text):
        mid = sha256(f"{tenant}:{user}:{text}".encode()).hexdigest()[:8]
        self.rows.append((mid, tenant, user, text))
        return mid
    def search(self, tenant, user, q):
        return [r for r in self.rows if r[1] == tenant and r[2] == user and q in r[3]]

api = TinyAPI()
api.add("t1", "u", "alpha")
api.add("t2", "u", "alpha-SECRET")
assert api.search("t1", "u", "SECRET") == []
assert len(api.search("t1", "u", "alpha")) == 1
print("project3 isolation smoke ok")


In [ ]:
# Journal morning brief formatter (Project 2)
brief = {
    "episodes": ["Gym", "Shipped PR"],
    "commitments": ["Send deck to Ava"],
}
print("# Morning brief")
print("## Recent")
for e in brief["episodes"]:
    print("-", e)
print("## Open commitments")
for c in brief["commitments"]:
    print("- [ ]", c)


### Try it yourself — Projects deepen

1. Add export endpoint: `GET /memories/export` returning JSON for a user.
2. Record three metrics after your demo: inject_rate, writes, forgets.


## Stretch Portfolio Ideas
- Browser extension that pins selected text into semantic memory
- Slack bot with per-workspace tenant isolation
- Memory debugger UI: show retrieved items + scores for each turn


## Key Takeaways

- Projects cement lifecycle + isolation habits
- Acceptance tests beat demo scripts for memory systems
- Start with prefs, then episodes, then a service boundary
